## 0. Librerías
Carga de dependencias para el pipeline de limpieza, unión e imputación.

In [1]:
import pandas as pd
import re
import os


## 1. Carga de datos
Lectura de las 4 bases SIES/Mi Futuro 2025-2026 (Carreras, Empleabilidad-Ingresos, Instituciones, Estadísticas por Carrera). `header=1` porque el Excel original trae una fila de títulos combinados (merged cells) antes de los nombres reales de columna.

In [2]:
carreras = pd.read_excel(
    r"Data\Raw\Buscador_de_Carreras_2025_2026_SIES_EEE.xlsx",
    sheet_name="Busc. Carreras  2025-2026",
    header=1,
)

empleabilidad = pd.read_excel(
    r"Data\Raw\Buscador_Empleabilidad_ingresos_2025_2026_SIES.xlsx",
    sheet_name="Carreras e IES (2025-2026)",
)

estadisticas_carrera = pd.read_excel(
    r"Data\Raw\Buscador_EstadísticasCarrera_2025_2026_SIES.xlsx",
    sheet_name="Hoja1",
    header=1,
)

instituciones = pd.read_excel(
    r"Data\Raw\Buscador_Instituciones_2025_2026_SIES-vf.xlsx",
    sheet_name="Buscador IES 25-26",
    header=1,
)

for name, df in [
    ("carreras", carreras),
    ("empleabilidad", empleabilidad),
    ("estadisticas_carrera", estadisticas_carrera),
    ("instituciones", instituciones),
]:
    print(name, df.shape)


carreras (9900, 39)
empleabilidad (1693, 13)
estadisticas_carrera (252, 53)
instituciones (124, 97)


## 2. Limpieza — `carreras`

Nivel: carrera × institución × sede (9.900 filas originales).

Reglas aplicadas:
- Nombres de columna sin espacios sobrantes.
- Eliminación de fila de footer (`FUENTE: ...`) y filas sin `Código único de carrera`.
- Columnas descartadas por baja utilidad para los KPIs definidos (`Rango ingreso a 1er año con PAES 2025`, `Promedio NEM 2025 de Matrícula 2025`).
- Nulos textuales (`-`, `s/i`) → `NA`.
- Arancel y costo de titulación: se separa la unidad (UF vs CLP) y se convierte todo a CLP usando **UF = $40.800**. Se conserva la unidad original en `arancel_unidad_original` / `titulacion_unidad_original` como columna de auditoría.
- Costo de titulación en $0: se deja tal cual (decisión: se asume que la institución no cobra ese trámite por separado, no que sea un dato faltante).
- Tipado: categóricas (`category`), identificadores (`string`), conteos y puntajes (`Int64`/`Float64` nullable, para soportar `NA` sin romper el tipo).
- Regla de negocio: `Vacantes 1er semestre` ≤ 5 se trata como dato no confiable → `NA` (para evitar falsos positivos de "alta demanda" con cupos casi nulos).

In [3]:
# --- Nombres de columna y filas basura ---
carreras.columns = carreras.columns.str.strip()

mask_footer = carreras.apply(
    lambda row: row.astype(str).str.contains("FUENTE", case=False, na=False).any(),
    axis=1
)
print("Filas footer detectadas:", mask_footer.sum())
carreras = carreras[~mask_footer]
carreras = carreras.dropna(subset=["Código único de carrera"]).reset_index(drop=True)

# --- Columnas descartadas ---
carreras = carreras.drop(columns=[
    "Rango ingreso a 1er año con PAES 2025",
    "Promedio NEM 2025 de Matrícula 2025",
])

# --- Nulos textuales -> NA ---
carreras = carreras.replace(
    to_replace=r"^\s*(-|s/i|S/I|S/i)\s*$", value=pd.NA, regex=True
)

# --- Arancel y Costo de titulación: separar unidad y convertir UF -> CLP ---
UF_VALOR = 40800

def split_currency(series):
    s = series.astype(str).str.strip()
    unidad = s.str.extract(r"(UF)", expand=False).fillna("CLP")
    valor = s.str.replace(r"[^\d]", "", regex=True).replace("", pd.NA)
    valor = pd.to_numeric(valor, errors="coerce")
    return unidad.where(series.notna()), valor

for col, prefix in [("Arancel Anual 2026", "arancel"), ("Costo de titulación", "titulacion")]:
    unidad, valor = split_currency(carreras[col])
    es_uf = unidad == "UF"
    valor_clp = valor.where(~es_uf, valor * UF_VALOR)
    carreras[f"{prefix}_unidad_original"] = unidad.astype("category")
    carreras[f"{prefix}_valor"] = valor_clp.astype("Int64")

carreras = carreras.drop(columns=["Arancel Anual 2026", "Costo de titulación"])

print(carreras["arancel_unidad_original"].value_counts(dropna=False))
print(carreras["titulacion_unidad_original"].value_counts(dropna=False))

# --- Categóricas ---
cat_cols = [
    "Área del conocimiento", "Tipo de institución", "Nombre institución",
    "Región", "Jornada", "Sede", "Nivel carrera", "Área Carrera Genérica",
]
carreras[cat_cols] = carreras[cat_cols].astype("category")

# --- Identificadores ---
carreras["Código único de carrera"] = carreras["Código único de carrera"].astype("string")
carreras["Código institución"] = carreras["Código institución"].astype("Int64").astype("string")

# --- Conteos ---
count_cols = [
    "Matrícula Total Femenina 2025", "Matrícula Total Masculina 2025", "Matrícula Total 2025",
    "Matrícula 1er año Femenina 2025", "Matrícula 1er año Masculina 2025", "Matrícula 1er año Total 2025",
    "Titulación Femenina 2024", "Titlación Masculina 2024", "Titulación Total 2024",
    "Vacantes 1er semestre", "Duración Formal (semestres)",
]
carreras[count_cols] = carreras[count_cols].apply(pd.to_numeric, errors="coerce").astype("Int64")

# --- Renombrar columnas de porcentaje ---
carreras = carreras.rename(columns={
    "Municipal y Servicios Locales": "Porcentaje Municipal",
    "Particular Subvencionado": "Porcentaje Subvencionado",
    "Particular Pagado": "Porcentaje Pagado",
    "Administración Delegada": "Porcentaje Admin Delegada",
})

# --- Porcentajes ---
pct_cols = ["Porcentaje Municipal", "Porcentaje Subvencionado", "Porcentaje Pagado", "Porcentaje Admin Delegada"]
carreras[pct_cols] = carreras[pct_cols].apply(pd.to_numeric, errors="coerce").astype("Float64")

# --- Puntajes PAES/NEM/Ranking ---
score_cols = ["NEM", "Ranking", "PAES Lenguaje", "PAES Matemáticas", "PAES Matemáticas 2",
              "PAES Historia", "PAES Ciencias", "Otros"]
carreras[score_cols] = carreras[score_cols].apply(pd.to_numeric, errors="coerce").astype("Float64")

# --- Promedio PAES 2025 ---
carreras["Promedio PAES 2025 de Matrícula 1er año 2025"] = pd.to_numeric(
    carreras["Promedio PAES 2025 de Matrícula 1er año 2025"], errors="coerce"
).astype("Float64")

# --- Regla de negocio: vacantes <=5 no confiables ---
carreras["Vacantes 1er semestre"] = carreras["Vacantes 1er semestre"].where(
    carreras["Vacantes 1er semestre"] > 5, pd.NA
)

carreras.info()
carreras.head()


Filas footer detectadas: 1
arancel_unidad_original
CLP    9785
UF      113
Name: count, dtype: int64
titulacion_unidad_original
CLP    9785
UF      113
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 9898 entries, 0 to 9897
Data columns (total 39 columns):
 #   Column                                        Non-Null Count  Dtype   
---  ------                                        --------------  -----   
 0   Código único de carrera                       9898 non-null   string  
 1   Código institución                            9898 non-null   string  
 2   Área del conocimiento                         9898 non-null   category
 3   Tipo de institución                           9898 non-null   category
 4   Nombre institución                            9898 non-null   category
 5   Nombre carrera                                9898 non-null   str     
 6   Región                                        9898 non-null   category
 7   Jornada                              

,Código único de carrera,Código institución,Área del conocimiento,Tipo de institución,Nombre institución,Nombre carrera,Región,Jornada,Sede,Duración Formal (semestres),...,PAES Matemáticas,PAES Matemáticas 2,PAES Historia,PAES Ciencias,Otros,Área Carrera Genérica,arancel_unidad_original,arancel_valor,titulacion_unidad_original,titulacion_valor
0,I1S1C10J4V1,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,LICENCIATURA EN HISTORIA,Metropolitana,A Distancia,CASA CENTRAL,8,...,0.0,0.0,0.0,0.0,0.0,Historia,CLP,3407000,CLP,321000
1,I1S1C10J4V2,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,LICENCIATURA EN HISTORIA,Metropolitana,A Distancia,CASA CENTRAL,4,...,0.0,0.0,0.0,0.0,0.0,Historia,CLP,3407000,CLP,340000
2,I1S1C10J4V3,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,LICENCIATURA EN HISTORIA,Metropolitana,A Distancia,CASA CENTRAL,5,...,0.0,0.0,0.0,0.0,0.0,Historia,CLP,3407000,CLP,340000
3,I1S1C12J1V1,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,PSICOLOGIA,Metropolitana,Diurna,CASA CENTRAL,10,...,10.0,0.0,10.0,10.0,0.0,Psicología,CLP,5629000,CLP,531000
4,I1S1C12J2V4,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,PSICOLOGIA,Metropolitana,Vespertina,CASA CENTRAL,8,...,0.0,0.0,0.0,0.0,0.0,Psicología,CLP,5629000,CLP,562000


## 3. Limpieza — `empleabilidad`

Nivel: carrera genérica × institución (1.690 combinaciones, según la metodología SIES).

Reglas aplicadas:
- Nulos textuales (`-`, `s/i`, `n/a`) → `NA`.
- Acreditación institucional: se separa en `acreditada` (bool) y `anios_acreditacion` (NA → 0, decisión: sin dato = no acreditada).
- **Ingreso promedio al 4° año**: viene como texto de tramo (`"De $1 millón 100 mil a $1 millón 200 mil"`). Se parsea a un valor numérico (punto medio del tramo). La función `parse_monto` combina correctamente "millón" + "mil" como un solo número compuesto — la primera versión de este parser tenía un bug que separaba esos dos componentes y los promediaba por error, generando valores absurdamente bajos (se corrigió tras detectarlo en la comparación pre/post imputación).
- Casos abiertos ("Sobre $3 millones 500 mil") se toman como el monto puntual mencionado, no como rango.
- Rename de columnas para alinear las llaves de unión con `carreras` (`Código` → `Código institución`, `Nombre carrera genérica` → `Área Carrera Genérica`).

In [4]:
# --- Nombres de columna y columna vacía final ---
empleabilidad.columns = empleabilidad.columns.str.strip()
empleabilidad = empleabilidad.loc[:, empleabilidad.columns.notna()]

# --- Filas basura ---
mask_footer = empleabilidad.apply(
    lambda row: row.astype(str).str.contains("FUENTE", case=False, na=False).any(),
    axis=1
)
print("Filas footer detectadas:", mask_footer.sum())
empleabilidad = empleabilidad[~mask_footer]
empleabilidad = empleabilidad.dropna(subset=["Código"]).reset_index(drop=True)

# --- Nulos textuales -> NA ---
empleabilidad = empleabilidad.replace(
    to_replace=r"^\s*(-|s/i|S/I|S/i|n/a|N/A)\s*$", value=pd.NA, regex=True
)

# --- Acreditación ---
col_acred = "Acreditación institución (al 31 de octubre 2025)"
empleabilidad["acreditada"] = ~empleabilidad[col_acred].isin(["No"])
empleabilidad["anios_acreditacion"] = pd.to_numeric(
    empleabilidad[col_acred].astype(str).str.extract(r"(\d+)", expand=False),
    errors="coerce"
).fillna(0).astype("Int64")
empleabilidad = empleabilidad.drop(columns=[col_acred])

# --- Parseo de tramo de ingreso (CORREGIDO: combina "millón" + "mil" como un solo monto) ---
def parse_monto(texto_monto):
    """Convierte 'X millón(es) Y mil' / 'Y mil' / 'X millones' a un número entero."""
    s = str(texto_monto).lower()
    total = 0
    m_millon = re.search(r"(\d+)\s*mill[oó]n(?:es)?", s)
    if m_millon:
        total += int(m_millon.group(1)) * 1_000_000
    m_mil = re.search(r"(\d+)\s*mil\b", s)
    if m_mil:
        total += int(m_mil.group(1)) * 1_000
    return total if total > 0 else None

def parse_tramo_ingreso(texto):
    if pd.isna(texto):
        return pd.NA
    texto = str(texto)

    # Caso abierto: "Sobre $3 millones 500 mil" -> se usa ese monto como estimador puntual
    if texto.strip().lower().startswith("sobre"):
        monto = parse_monto(texto)
        return monto if monto else pd.NA

    # Caso rango: "De/Desde $X a $Y" -> separar por " a " y promediar los dos montos completos
    partes = re.split(r"\s+a\s+", texto, maxsplit=1)
    if len(partes) == 2:
        inferior = parse_monto(partes[0])
        superior = parse_monto(partes[1])
        if inferior and superior:
            return (inferior + superior) / 2
        return inferior or superior or pd.NA

    monto_unico = parse_monto(texto)
    return monto_unico if monto_unico else pd.NA

col_ingreso = "Ingreso Promedio al 4° año"
empleabilidad["ingreso_4to_anio_valor"] = empleabilidad[col_ingreso].apply(parse_tramo_ingreso).astype("Float64")
empleabilidad = empleabilidad.rename(columns={col_ingreso: "ingreso_4to_anio_tramo"})

# --- Categóricas ---
cat_cols = ["Tipo de institución", "Nombre de institución", "Área", "Nombre carrera genérica"]
empleabilidad[cat_cols] = empleabilidad[cat_cols].astype("category")

# --- Identificador ---
empleabilidad["Código"] = empleabilidad["Código"].astype("Int64").astype("string")

# --- Numéricas ---
num_cols = [
    "% titulados con continuidad de estudios", "Retención 1er\xa0año", "Duración Real (semestres)",
    "Empleabilidad 1er año", "Empleabilidad 2° año",
]
empleabilidad[num_cols] = empleabilidad[num_cols].apply(pd.to_numeric, errors="coerce").astype("Float64")

# --- Rename final para alinear llaves con `carreras` ---
empleabilidad = empleabilidad.rename(columns={
    "Código": "Código institución",
    "Nombre carrera genérica": "Área Carrera Genérica",
    "Retención 1er\xa0año": "Retención 1er año",
})

empleabilidad.info()
empleabilidad.head()


Filas footer detectadas: 1
<class 'pandas.DataFrame'>
RangeIndex: 1690 entries, 0 to 1689
Data columns (total 15 columns):
 #   Column                                   Non-Null Count  Dtype   
---  ------                                   --------------  -----   
 0   Código institución                       1690 non-null   string  
 1   Tipo de institución                      1690 non-null   category
 2   Nombre de institución                    1690 non-null   category
 3   Área                                     1690 non-null   category
 4   Área Carrera Genérica                    1690 non-null   category
 5   Nombre carrera (del título)              1690 non-null   str     
 6   % titulados con continuidad de estudios  1690 non-null   Float64 
 7   Retención 1er año                        1411 non-null   Float64 
 8   Duración Real (semestres)                1296 non-null   Float64 
 9   Empleabilidad 1er año                    1690 non-null   Float64 
 10  Empleabilidad 2° año

,Código institución,Tipo de institución,Nombre de institución,Área,Área Carrera Genérica,Nombre carrera (del título),% titulados con continuidad de estudios,Retención 1er año,Duración Real (semestres),Empleabilidad 1er año,Empleabilidad 2° año,ingreso_4to_anio_tramo,acreditada,anios_acreditacion,ingreso_4to_anio_valor
0,701,Centros de Formación Técnica,CFT MANPOWER,Administración y Comercio,Secretariado Bilingüe,Asistente Ejecutivo Bilingüe o Asistente Ejecu...,0.088757,0.652893,7.235294,0.71875,0.705036,De $1 millón a $1 millón 100 mil,True,3,1050000.0
1,902,Centros de Formación Técnica,CFT DE LA REGION DE ARICA Y PARINACOTA,Agropecuaria,Técnico Agropecuario,Técnico de Nivel Superior Agrícola,0.382609,0.679487,5.405405,0.620253,<NA>,NaN,False,0,<NA>
2,430,Centros de Formación Técnica,CFT INACAP,Agropecuaria,Técnico Agropecuario,Tecnología Agrícola y Producción Ganadera,0.208835,0.666038,7.748111,0.475706,0.53629,De $900 mil a $1 millón,True,7,950000.0
3,629,Centros de Formación Técnica,CFT PUCV,Agropecuaria,Técnico Agropecuario,Técnico de Nivel Superior en Agrícola,0.113402,0.810811,5.964286,0.559322,<NA>,NaN,True,5,<NA>
4,367,Centros de Formación Técnica,CFT SAN AGUSTIN,Agropecuaria,Técnico Agropecuario,Técnico Agrícola,0.076046,0.629808,5.838462,0.380645,0.476471,De $800 mil a $900 mil,True,5,850000.0


## 4. Limpieza — `instituciones`

Nivel: institución (121 filas).

Se descartan bloques no usados por los KPIs definidos (financiero, infraestructura, JCE/personal académico, posgrado) para mantener el dataset enfocado. Si más adelante se necesita alguno, se recupera directo del Excel original.

Reglas aplicadas:
- Nulos textuales → `NA`.
- Acreditación institucional: `anios_acreditacion` (NA → 0) y `acreditada` (bool).
- Selección de columnas relevantes: identificación, ubicación, acreditación, matrícula/titulados de pregrado histórico, retención, puntajes de ingreso promedio, duración formal/real.

In [5]:
# --- Nombres de columna y filas basura ---
instituciones.columns = instituciones.columns.str.strip()

mask_footer = instituciones.apply(
    lambda row: row.astype(str).str.contains("FUENTE", case=False, na=False).any(),
    axis=1
)
print("Filas footer detectadas:", mask_footer.sum())
instituciones = instituciones[~mask_footer]
instituciones = instituciones.dropna(subset=["Código institución"]).reset_index(drop=True)

# --- Nulos textuales -> NA ---
instituciones = instituciones.replace(
    to_replace=r"^\s*(-|s/i|S/I|S/i|n/a|N/A)\s*$", value=pd.NA, regex=True
)

# --- Acreditación institucional ---
instituciones["anios_acreditacion"] = pd.to_numeric(
    instituciones["Años acreditación (30 de octubre de 2025)"], errors="coerce"
).fillna(0).astype("Int64")
instituciones["acreditada"] = instituciones["Acreditación (30 de octubre de 2025)"] != "No"

# --- Identificador ---
instituciones["Código institución"] = instituciones["Código institución"].astype("Int64").astype("string")

# --- Selección de columnas relevantes ---
cols_mantener = [
    "Código institución", "Tipo de institución", "Nombre institución", "Autonomía",
    "Dirección Sede Central", "Página web",
    "acreditada", "anios_acreditacion",
    "Matrícula Pregrado 2020", "Matrícula Pregrado 2021", "Matrícula Pregrado 2022",
    "Matrícula Pregrado 2023", "Matrícula Pregrado 2024", "Matrícula Pregrado 2025",
    "Retención 1er año (acorde a metodología histórica del buscador)", "Retención 2° año",
    "Promedio NEM matriculados 1er año 2025", "Promedio PAES matriculados 1er año 2025",
    "Titulados Pregrado 2020", "Titulados Pregrado 2021", "Titulados Pregrado 2022",
    "Titulados Pregrado 2023", "Titulados Pregrado 2024",
    "Duración Formal", "Duración Real",
]
instituciones = instituciones[cols_mantener]

# --- Categóricas ---
cat_cols = ["Tipo de institución", "Nombre institución", "Autonomía"]
instituciones[cat_cols] = instituciones[cat_cols].astype("category")

# --- Numéricas ---
num_cols = [c for c in instituciones.columns if c not in cat_cols + ["Código institución", "Dirección Sede Central", "Página web", "acreditada"]]
instituciones[num_cols] = instituciones[num_cols].apply(pd.to_numeric, errors="coerce").astype("Float64")
instituciones["anios_acreditacion"] = instituciones["anios_acreditacion"].astype("Int64")

# --- Rename para consistencia ---
instituciones = instituciones.rename(columns={
    "Retención 1er año (acorde a metodología histórica del buscador)": "Retención 1er año",
})

instituciones.info()
instituciones.head()


Filas footer detectadas: 1
<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 25 columns):
 #   Column                                   Non-Null Count  Dtype   
---  ------                                   --------------  -----   
 0   Código institución                       121 non-null    string  
 1   Tipo de institución                      121 non-null    category
 2   Nombre institución                       121 non-null    category
 3   Autonomía                                121 non-null    category
 4   Dirección Sede Central                   121 non-null    str     
 5   Página web                               121 non-null    str     
 6   acreditada                               121 non-null    bool    
 7   anios_acreditacion                       121 non-null    Int64   
 8   Matrícula Pregrado 2020                  121 non-null    Float64 
 9   Matrícula Pregrado 2021                  121 non-null    Float64 
 10  Matrícula Pregrado 202

,Código institución,Tipo de institución,Nombre institución,Autonomía,Dirección Sede Central,Página web,acreditada,anios_acreditacion,Matrícula Pregrado 2020,Matrícula Pregrado 2021,...,Retención 2° año,Promedio NEM matriculados 1er año 2025,Promedio PAES matriculados 1er año 2025,Titulados Pregrado 2020,Titulados Pregrado 2021,Titulados Pregrado 2022,Titulados Pregrado 2023,Titulados Pregrado 2024,Duración Formal,Duración Real
0,1,Universidades,Universidad Gabriela Mistral,Autónoma,"Av. Ricardo Lyon Nº 1177 - Providencia, Santiago",www.ugm.cl,True,4,959.0,1193.0,...,0.674718,5.856716,579.436416,368.0,301.0,273.0,272.0,339.0,9.372881,12.220339
1,2,Universidades,Universidad Finis Terrae,Autónoma,"Av. Pedro de Valdivia Nº 1509 - Providencia, S...",www.finisterrae.cl,True,5,8040.0,8475.0,...,0.728175,6.194287,638.348548,742.0,1161.0,1054.0,1073.0,1008.0,10.033846,11.969231
2,3,Universidades,Universidad Diego Portales,Autónoma,"Av. Manuel Rodríguez Sur Nº 415, Santiago",www.udp.cl,True,6,16269.0,16759.0,...,0.814534,6.346192,701.070518,1498.0,2939.0,2673.0,2512.0,2870.0,9.710958,12.021
3,4,Universidades,Universidad Central de Chile,Autónoma,"Toesca Nº 1783, Santiago",www.ucentral.cl,True,5,11748.0,11827.0,...,0.776778,6.139515,603.329661,1956.0,2147.0,1972.0,2712.0,2293.0,10.041497,12.387075
4,7,Universidades,Universidad Bolivariana,Autónoma,"Huérfanos Nº 1721 , Santiago",www.ubolivariana.cl,True,0,3144.0,2431.0,...,<NA>,<NA>,<NA>,688.0,1823.0,759.0,462.0,323.0,10.253623,15.217391


## 5. Limpieza — `estadisticas_carrera`

Nivel: carrera genérica × **tipo de institución** (agregado, 252 combinaciones).

Esta base se usa como **fuente de relleno real** para `Empleabilidad`, `Retención` e `ingreso_4to_anio` cuando el dato específico por institución no existe (ver sección 7) — no se usa para cálculos independientes en el dashboard, solo como respaldo del dataset maestro.

Reglas aplicadas: mismo patrón de limpieza de nulos, tipado de identificador/categóricas/numéricas, y corrección de un typo de origen en el nombre de columna (`Partiular` → `Particular`).

In [6]:
# --- Nombres de columna ---
estadisticas_carrera.columns = estadisticas_carrera.columns.str.strip()

# --- Filas basura ---
mask_footer = estadisticas_carrera.apply(
    lambda row: row.astype(str).str.contains("FUENTE", case=False, na=False).any(),
    axis=1
)
print("Filas footer detectadas:", mask_footer.sum())
estadisticas_carrera = estadisticas_carrera[~mask_footer]
estadisticas_carrera = estadisticas_carrera.dropna(subset=["ID"]).reset_index(drop=True)

# --- Nulos textuales -> NA ---
estadisticas_carrera = estadisticas_carrera.replace(
    to_replace=r"^\s*(-|s/i|S/I|S/i|n/a|N/A)\s*$", value=pd.NA, regex=True
)

# --- Identificador ---
estadisticas_carrera["ID"] = estadisticas_carrera["ID"].astype("Int64").astype("string")

# --- Categóricas ---
cat_cols = ["Área", "Tipo de institución", "Carrera genérica"]
estadisticas_carrera[cat_cols] = estadisticas_carrera[cat_cols].astype("category")

# --- Numéricas ---
num_cols = [c for c in estadisticas_carrera.columns if c not in cat_cols + ["ID"]]
estadisticas_carrera[num_cols] = estadisticas_carrera[num_cols].apply(pd.to_numeric, errors="coerce").astype("Float64")

# --- Rename para alinear llaves y corregir typo de origen ---
estadisticas_carrera = estadisticas_carrera.rename(columns={
    "Carrera genérica": "Área Carrera Genérica",
    "Partiular Subvencionado": "Particular Subvencionado",
})

estadisticas_carrera.info()
estadisticas_carrera.head()


Filas footer detectadas: 0
<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 53 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   ID                             252 non-null    string  
 1   Área                           252 non-null    category
 2   Tipo de institución            252 non-null    category
 3   Área Carrera Genérica          252 non-null    category
 4   1er año                        252 non-null    Float64 
 5   2° año                         252 non-null    Float64 
 6   3er año                        252 non-null    Float64 
 7   4° año                         252 non-null    Float64 
 8   5° año                         252 non-null    Float64 
 9   10% inferior 1er año           252 non-null    Float64 
 10  25% inferior 1er año           252 non-null    Float64 
 11  Percentil 50 1er año           252 non-null    Float64 
 12  25% superior 1er año

,ID,Área,Tipo de institución,Área Carrera Genérica,1er año,2° año,3er año,4° año,5° año,10% inferior 1er año,...,Total Matrícula 1er año,Matrícula Total Mujeres,Matrícula Total Hombres,Matrícula Total,Retención 1er año,Retención 2° año,Municipal y Servicios Locales,Particular Subvencionado,Particular Pagado,Administración Delegada
0,1,Administración y Comercio,Centros de Formación Técnica,Secretariado Bilingüe,1031117.626953,1096161.371712,1113420.142315,1136314.129544,1166457.517424,662930.262083,...,138.0,271.0,27.0,298.0,0.6375,<NA>,0.208531,0.630332,0.132701,0.028436
1,2,Agropecuaria,Centros de Formación Técnica,Técnico Agropecuario,847957.331796,884213.361134,935086.079927,997009.39983,1031960.969146,551775.462083,...,1428.0,1426.0,1639.0,3065.0,0.664901,<NA>,0.489354,0.419092,0.021647,0.069908
2,3,Educación,Centros de Formación Técnica,Técnico Asistente del Educador de Párvulos,644643.561025,669809.430528,677571.671355,688296.479653,711830.198268,524986.286667,...,3985.0,10206.0,30.0,10236.0,0.725403,<NA>,0.518772,0.438715,0.00519,0.037323
3,4,Educación,Centros de Formación Técnica,Técnico Asistente del Educador Diferencial,620472.528346,646878.660759,655863.605785,655973.660806,708505.666179,524841.048333,...,1715.0,4414.0,156.0,4570.0,0.741803,<NA>,0.525709,0.436168,0.00627,0.031854
4,5,Salud,Centros de Formación Técnica,Técnico Dental y Asistente de Odontología,657951.291473,694369.972292,725072.844965,764439.793273,801109.646783,533958.094583,...,1093.0,2478.0,392.0,2870.0,0.72093,<NA>,0.47061,0.48923,0.005111,0.035049


## 6. Unión — dataset analítico maestro

`carreras` (base) ⟵ `instituciones` (por `Código institución`) ⟵ `empleabilidad` (por `Código institución` + `Área Carrera Genérica`).

`estadisticas_carrera` queda fuera de esta unión (ver sección 5).

In [7]:
dataset = carreras.merge(
    instituciones[["Código institución", "acreditada", "anios_acreditacion",
                   "Dirección Sede Central", "Página web"]],
    on="Código institución", how="left"
)

dataset = dataset.merge(
    empleabilidad[[
        "Código institución", "Área Carrera Genérica",
        "Empleabilidad 1er año", "Empleabilidad 2° año", "ingreso_4to_anio_valor",
        "Retención 1er año", "Duración Real (semestres)",
        "% titulados con continuidad de estudios",
    ]],
    on=["Código institución", "Área Carrera Genérica"], how="left"
)

print(dataset.shape)
dataset.info()


(9898, 49)
<class 'pandas.DataFrame'>
RangeIndex: 9898 entries, 0 to 9897
Data columns (total 49 columns):
 #   Column                                        Non-Null Count  Dtype   
---  ------                                        --------------  -----   
 0   Código único de carrera                       9898 non-null   string  
 1   Código institución                            9898 non-null   string  
 2   Área del conocimiento                         9898 non-null   category
 3   Tipo de institución                           9898 non-null   category
 4   Nombre institución                            9898 non-null   category
 5   Nombre carrera                                9898 non-null   str     
 6   Región                                        9898 non-null   category
 7   Jornada                                       9898 non-null   category
 8   Sede                                          9898 non-null   category
 9   Duración Formal (semestres)                   9898 n

## 7. Relleno con datos agregados reales de SIES

**No se usa imputación estadística (KNN).** En su lugar, cuando falta el dato específico de una combinación carrera-institución en `Empleabilidad 1er/2° año`, `Retención 1er año`, `ingreso_4to_anio_valor` o `Duración Real`, se rellena con el valor real que SIES publica a un nivel más agregado (carrera genérica × tipo de institución, en `estadisticas_carrera`) — es sustitución por dato oficial, no una estimación basada en variables ajenas (arancel, puntajes, etc.).

Se guarda una bandera `_relleno_agregado` por columna para dejar trazabilidad de qué valores son el dato específico observado vs. el agregado usado como respaldo. Lo que siga en `NA` después de este paso es porque el dato tampoco existe a nivel agregado (ej. carreras genéricas excluidas de la metodología SIES, como Historia o Filosofía) — se deja como `NA` real, sin inventar nada.

In [8]:
mapeo_columnas = {
    "Empleabilidad 1er año": "Empleabilidad 1er año",
    "Empleabilidad 2° año": "Empleabilidad 2° año",
    "Retención 1er año": "Retención 1er año",
    "ingreso_4to_anio_valor": "4° año",
    "Duración Real (semestres)": "Duración Real",
}

lookup_agregado = estadisticas_carrera.set_index(
    ["Área Carrera Genérica", "Tipo de institución"]
)[list(mapeo_columnas.values())]

llave_dataset = pd.MultiIndex.from_frame(dataset[["Área Carrera Genérica", "Tipo de institución"]])

for col_dataset, col_agregado in mapeo_columnas.items():
    dataset[f"{col_dataset}_relleno_agregado"] = dataset[col_dataset].isna()  # bandera: vino del agregado

    valores_agregado = llave_dataset.map(lookup_agregado[col_agregado])
    dataset[col_dataset] = dataset[col_dataset].fillna(pd.Series(valores_agregado.values, index=dataset.index))

# Reaplicar regla de negocio de vacantes (por si el join la sobrescribió)
dataset["Vacantes 1er semestre"] = dataset["Vacantes 1er semestre"].where(
    dataset["Vacantes 1er semestre"] > 5, pd.NA
)

dataset.info()


<class 'pandas.DataFrame'>
RangeIndex: 9898 entries, 0 to 9897
Data columns (total 54 columns):
 #   Column                                        Non-Null Count  Dtype   
---  ------                                        --------------  -----   
 0   Código único de carrera                       9898 non-null   string  
 1   Código institución                            9898 non-null   string  
 2   Área del conocimiento                         9898 non-null   category
 3   Tipo de institución                           9898 non-null   category
 4   Nombre institución                            9898 non-null   category
 5   Nombre carrera                                9898 non-null   str     
 6   Región                                        9898 non-null   category
 7   Jornada                                       9898 non-null   category
 8   Sede                                          9898 non-null   category
 9   Duración Formal (semestres)                   9898 non-null   I

## 8. Validación del relleno

Comparación de valores faltantes antes/después del relleno con datos agregados de SIES. `faltantes_despues` puede ser mayor a 0: son carreras que tampoco tienen dato a nivel agregado, y quedan como `NA` real.

In [9]:
cols_rellenadas = list(mapeo_columnas.keys())

resumen = pd.DataFrame({
    "faltantes_antes": [dataset[f"{col}_relleno_agregado"].sum() for col in cols_rellenadas],
    "total_filas": len(dataset),
})
resumen.index = cols_rellenadas
resumen["% faltante_antes"] = (resumen["faltantes_antes"] / resumen["total_filas"] * 100).round(1)
resumen["faltantes_despues"] = [dataset[col].isna().sum() for col in cols_rellenadas]

resumen


,faltantes_antes,total_filas,% faltante_antes,faltantes_despues
Empleabilidad 1er año,3700,9898,37.4,819
Empleabilidad 2° año,4029,9898,40.7,832
Retención 1er año,3903,9898,39.4,864
ingreso_4to_anio_valor,4881,9898,49.3,925
Duración Real (semestres),4208,9898,42.5,896


## 9. Redondeo

Se redondean a 2 decimales todas las columnas numéricas de tipo decimal (`Float64`), tanto en `dataset` como en `estadisticas_carrera`. Se hace **después** de la imputación KNN, para no perder los decimales que el modelo calcula al promediar vecinos.

In [10]:
float_cols_dataset = dataset.select_dtypes(include=["Float64", "float64"]).columns
dataset[float_cols_dataset] = dataset[float_cols_dataset].round(2)

float_cols_estad = estadisticas_carrera.select_dtypes(include=["Float64", "float64"]).columns
estadisticas_carrera[float_cols_estad] = estadisticas_carrera[float_cols_estad].round(2)

dataset.head()


,Código único de carrera,Código institución,Área del conocimiento,Tipo de institución,Nombre institución,Nombre carrera,Región,Jornada,Sede,Duración Formal (semestres),...,Empleabilidad 2° año,ingreso_4to_anio_valor,Retención 1er año,Duración Real (semestres),% titulados con continuidad de estudios,Empleabilidad 1er año_relleno_agregado,Empleabilidad 2° año_relleno_agregado,Retención 1er año_relleno_agregado,ingreso_4to_anio_valor_relleno_agregado,Duración Real (semestres)_relleno_agregado
0,I1S1C10J4V1,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,LICENCIATURA EN HISTORIA,Metropolitana,A Distancia,CASA CENTRAL,8,...,<NA>,<NA>,<NA>,<NA>,<NA>,True,True,True,True,True
1,I1S1C10J4V2,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,LICENCIATURA EN HISTORIA,Metropolitana,A Distancia,CASA CENTRAL,4,...,<NA>,<NA>,<NA>,<NA>,<NA>,True,True,True,True,True
2,I1S1C10J4V3,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,LICENCIATURA EN HISTORIA,Metropolitana,A Distancia,CASA CENTRAL,5,...,<NA>,<NA>,<NA>,<NA>,<NA>,True,True,True,True,True
3,I1S1C12J1V1,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,PSICOLOGIA,Metropolitana,Diurna,CASA CENTRAL,10,...,0.71,1450000.0,0.9,11.5,0.0,False,False,False,False,True
4,I1S1C12J2V4,1,Ciencias Sociales,Universidades,UNIVERSIDAD GABRIELA MISTRAL,PSICOLOGIA,Metropolitana,Vespertina,CASA CENTRAL,8,...,0.71,1450000.0,0.9,11.5,0.0,False,False,False,False,True


## 10. Exportar dataset limpio

Se guarda en `Data\Clean\`, en `.csv` (para inspección rápida / Excel) y `.parquet` (conserva los tipos de dato exactos, recomendado para seguir trabajando en Python).

In [11]:
os.makedirs("Data\\Clean", exist_ok=True)

dataset.to_csv("Data\\Clean\\dataset_analitico_maestro.csv", index=False, encoding="utf-8-sig")
dataset.to_parquet("Data\\Clean\\dataset_analitico_maestro.parquet", index=False)

estadisticas_carrera.to_csv("Data\\Clean\\estadisticas_carrera.csv", index=False, encoding="utf-8-sig")
estadisticas_carrera.to_parquet("Data\\Clean\\estadisticas_carrera.parquet", index=False)

print("dataset:", dataset.shape)
print("estadisticas_carrera:", estadisticas_carrera.shape)


dataset: (9898, 54)
estadisticas_carrera: (252, 53)


## 11. Agregación a carrera genérica (base del dashboard)

El dashboard trabaja a nivel de **carrera genérica** (una fila por carrera, 165 filas), consolidando universidades, IP y CFT. Reglas de agregación:

- **Indicadores de resultado** (ingresos 1°-5° año, tramos, empleabilidad y su evolución, retención, duración): promedio **ponderado por titulados** de cada tipo de institución, desde `estadisticas_carrera` (252 combinaciones oficiales SIES).
- **Matrícula, titulados**: suma directa.
- **Arancel, costo de titulación, puntaje de corte PAES**: promedio **ponderado por matrícula** de los 9.898 programas del dataset maestro.
- **Vacantes y matrícula 1er año**: se calculan en versión total y **presencial** (excluyendo jornada "A Distancia"), porque las carreras online inflan estos valores (~15% de la matrícula nacional es a distancia, y algunas carreras son 100% online). Se guarda además `pct_matricula_distancia` como variable de transparencia.

Exporta `Data\Clean\dataset_carrera_generica.parquet`, que es la base que consume el dashboard.

In [ ]:
import numpy as np

def _wmean(valores, pesos):
    """Promedio ponderado que ignora NA; cae a promedio simple si los pesos suman 0."""
    mask = valores.notna()
    v, w = valores[mask].astype(float), pesos[mask].astype(float)
    if len(v) == 0:
        return np.nan
    if w.sum() == 0:
        return float(v.mean())
    return float(np.average(v, weights=w))


KEY = "Área Carrera Genérica"

# --- Paso 1: colapsar estadisticas_carrera (252 combos) a carrera genérica ---
est = estadisticas_carrera.copy()
for c in est.select_dtypes(include=["Float64", "Int64"]).columns:
    est[c] = est[c].astype(float)
for k in [KEY, "Área", "Tipo de institución"]:
    if str(est[k].dtype) == "category":
        est[k] = est[k].astype(str)

cols_ingresos = ["1er año", "2° año", "3er año", "4° año", "5° año"]
cols_tramos = [c for c in est.columns if "inferior" in c or "superior" in c or "Percentil" in c]
cols_evo_emp = [c for c in est.columns if c.startswith("Empleabilidad") and " - " in c]
cols_wmean = cols_ingresos + cols_tramos + cols_evo_emp + [
    "Empleabilidad 1er año", "Empleabilidad 2° año", "Duración Formal", "Duración Real",
    "Retención 1er año", "Retención 2° año",
]
cols_sum = [
    "Titulados Mujeres", "Titulados Hombres", "Titulados Total",
    "Matrícula 1er año Mujeres", "Matrícula 1er año Hombres", "Total Matrícula 1er año",
    "Matrícula Total Mujeres", "Matrícula Total Hombres", "Matrícula Total",
]

filas = []
for carrera, g in est.groupby(KEY, observed=True):
    fila = {
        KEY: carrera,
        "Área": g["Área"].mode().iat[0],
        "tipos_institucion": " / ".join(sorted(g["Tipo de institución"].unique())),
        "n_tipos_institucion": g["Tipo de institución"].nunique(),
    }
    for c in cols_wmean:
        fila[c] = _wmean(g[c], g["Titulados Total"])
    for c in cols_sum:
        fila[c] = pd.to_numeric(g[c], errors="coerce").sum()
    filas.append(fila)
dataset_generica = pd.DataFrame(filas)

# --- Paso 2: agregados del maestro (9.898 programas) por carrera genérica ---
peso = "Matrícula Total 2025"
filas_m = []
for carrera, g in dataset.groupby(KEY, observed=True):
    pres = g[g["Jornada"] != "A Distancia"]  # solo jornadas presenciales/semipresenciales
    filas_m.append({
        KEY: carrera,
        "arancel_valor": _wmean(g["arancel_valor"], g[peso]),
        "titulacion_valor": _wmean(g["titulacion_valor"], g[peso]),
        "puntaje_corte_paes": _wmean(g["Promedio PAES 2025 de Matrícula 1er año 2025"], g[peso]),
        "vacantes_total": pd.to_numeric(g["Vacantes 1er semestre"], errors="coerce").sum(),
        "vacantes_presencial": pd.to_numeric(pres["Vacantes 1er semestre"], errors="coerce").sum(),
        "matricula_1er_total": pd.to_numeric(g["Matrícula 1er año Total 2025"], errors="coerce").sum(),
        "matricula_1er_presencial": pd.to_numeric(pres["Matrícula 1er año Total 2025"], errors="coerce").sum(),
        "pct_matricula_distancia": (
            1 - pd.to_numeric(pres[peso], errors="coerce").sum()
            / max(pd.to_numeric(g[peso], errors="coerce").sum(), 1)
        ),
        "prop_programas_acreditados": g["acreditada"].mean(),
        "n_programas": len(g),
        "n_instituciones": g["Nombre institución"].nunique(),
    })
dataset_generica = dataset_generica.merge(pd.DataFrame(filas_m), on=KEY, how="left")

# --- Variables derivadas ---
dataset_generica["carrera_tipo"] = dataset_generica[KEY]
dataset_generica["ingreso_4to_anio_valor"] = dataset_generica["4° año"]
dataset_generica["brecha_duracion"] = dataset_generica["Duración Real"] - dataset_generica["Duración Formal"]
dataset_generica["costo_total_carrera"] = (
    dataset_generica["arancel_valor"] * (dataset_generica["Duración Formal"] / 2)
    + dataset_generica["titulacion_valor"]
)
dataset_generica["anios_recuperar_inversion"] = (
    dataset_generica["costo_total_carrera"] / (dataset_generica["ingreso_4to_anio_valor"] * 12)
).round(1)

print("dataset_carrera_generica:", dataset_generica.shape)
dataset_generica.head()

In [ ]:
dataset_generica.to_parquet("Data\\Clean\\dataset_carrera_generica.parquet", index=False)
dataset_generica.to_csv("Data\\Clean\\dataset_carrera_generica.csv", index=False, encoding="utf-8-sig")
print("Exportado Data\\Clean\\dataset_carrera_generica.parquet")